In [1]:
# Importar librerias 
# Importar librerias con parche de entorno virtual
import os
import sys

# Mirar dentro entorno virtual asignado
ruta_venv = os.path.abspath(".venv/lib/site-packages")
if ruta_venv not in sys.path:
    sys.path.insert(0, ruta_venv)

import argparse
import json
import time
from pathlib import Path

import requests
import geopandas as gpd
from shapely.geometry import shape

In [2]:
# Overpass API endpoint (use mirror if rate-limited)
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OVERPASS_MIRROR = "https://overpass.kumi.systems/api/interpreter"

QUERY_TIME_OUT = 600  # seconds
HEADERS = {
    "User-Agent": "CityPedestrianZones/1.0 (urban-pedestrian-connectivity-research; md.hoyos729@gmail.com)",
    "Accept": "application/json",
}

In [3]:
def build_query(
    city: str | None = None,
    country: str | None = None,
    bbox: tuple[float, float, float, float] | None = None,
    time_out: int = 600,
) -> str:
    """
    Build an Overpass query based on available location inputs.
    Priority: bbox > city (+ optional country)
    """
    if bbox:
        south, west, north, east = bbox
        spatial_filter = f"({south},{west},{north},{east})"
        area_setup = ""
    elif city:
        area_name = f"{city},{country}" if country else city
        area_setup = f'{{{{geocodeArea:{area_name}}}}}->.searchArea;\n'
        spatial_filter = "(area.searchArea)"
    else:
        raise ValueError("Must provide either bbox or city.")

    return f"""
    [out:json][timeout:{time_out}];
    {area_setup}(
        way["highway"="pedestrian"]["bridge"!~"."]{spatial_filter};
        way["highway"="footway"]["footway"!~"^(sidewalk|crossing|traffic_island|link)$"]["bridge"!~"."]{spatial_filter};
        way["highway"="path"]["footway"!~"^(sidewalk|crossing)$"]["bridge"!~"."]{spatial_filter};
        way["leisure"="park"]{spatial_filter};
        relation["leisure"="park"]{spatial_filter};
    );
    out body geom;
    """

In [4]:
def run_query(query: str, time_out: int=600, header: str=None, retries: int=3) -> dict:
    """POST query to Overpass API with retry logic."""
    for attempt in range(retries):
        try:
            print(f"[query] Sending Overpass query (attempt {attempt + 1}/{retries})...")
            response = requests.post(
                OVERPASS_URL,
                data={"data": query},
                timeout=time_out,
                headers=header if header else None
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            print(f"[query] Timeout — retrying in 10s...")
            time.sleep(10)
        except requests.exceptions.HTTPError as e:
            if response.status_code == 400:
                print(f"[query] Bad request — check your query syntax.")
                raise e
            elif response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 60))
                print(f"[query] Rate limited — waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"[query] HTTP error — {response.status_code}")
                raise e

    raise RuntimeError("Overpass query failed after all retries.")

In [5]:
def overpass_to_geojson(data: dict) -> dict:
    """
    Convert Overpass JSON response to GeoJSON FeatureCollection.
    Each OSM way becomes a LineString Feature with all tags as properties.
    """
    features = []

    for element in data.get("elements", []):
        # Security check
        if element["type"] not in ["way", "relation"]:
            continue

        # Build geometry from node coordinates
        coords = [
            [node["lon"], node["lat"]]
            for node in element.get("geometry", [])
        ]

        # Skip features without valid geometry (a node or an element with missing geometry)
        if len(coords) < 3:
            continue

        tags = element.get("tags", {})

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords,
            },
            "properties": {
                "osm_id": element["id"],
                "tipoobjeto": element["type"], # ¿Es línea?
                
                # Identificación
                "nombre": tags.get("name", "Sin nombre"),
                "categoria": tags.get("highway") if tags.get("highway") else tags.get("leisure"),

                # Características Peatonales
                "superficie": tags.get("surface"),      # ¿Es cemento, pasto o tierra?
                "ancho": tags.get("width"),             # ¿Es un sendero ancho o angosto?
                "iluminacion": tags.get("lit"),         # ¿Es seguro de noche? (yes/no)
                "acceso": tags.get("access"),           # ¿Es público o privado?
                "bicicletas": tags.get("bicycle"),      # ¿Se permiten bicis?
                "sillasruedas": tags.get("wheelchair"), # ¿Es accesible para todos?
                
                # Para saber si es parque o calle peatonal
                "es_parque": "Sí" if tags.get("leisure") == "park" else "No",
                "tiposendero": tags.get("footway"),    # ¿Es andén, cruce o sendero de parque?
                
                # Raw tags for reference
                "_all_tags": json.dumps(tags, ensure_ascii=False),
            },
        }
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features,
    }
    return geojson

In [6]:
def identificar_zonas_interrumpidas(zonas_geojson, radio_metros=20):
    """
    Examina la proximidad de cada zona peatonal/parque.
    Si a menos de 'radio_metros' (20m) existe OTRA zona peatonal,
    significa que el espacio está interrumpido pero mantiene potencial de continuidad.
    """
    features = zonas_geojson["features"]
    elementos_validos = []
    
    # Aproximación directa muy eficiente en grados para Bogotá (1 metro ≈ 0.000009 grados)
    CONV_METROS = 0.000009
    radio_grados = radio_metros * CONV_METROS
    
    # Convertimos las geometrías de GeoJSON a objetos de Shapely
    geometrias = []
    for f in features:
        try:
            geom_shapely = shape(f["geometry"])
            if geom_shapely.is_valid:
                geometrias.append((f["properties"]["osm_id"], geom_shapely, f))
        except Exception:
            continue

    print(f"[Análisis] Procesando {len(geometrias)} elementos peatonales")

    # Comparar cada elemento con todos los demás usando un área de 20 metros
    for i, (id_actual, geom_actual, feature_actual) in enumerate(geometrias):
        tiene_vecino_cercano = False
        zona_influencia = geom_actual.buffer(radio_grados)
        
        for j, (id_otro, geom_otro, _) in enumerate(geometrias):
            if i == j:
                continue 
                
            if zona_influencia.intersects(geom_otro):
                tiene_vecino_cercano = True
                break 
        
        # Clasificación con tus nuevas etiquetas
        if tiene_vecino_cercano:
            feature_actual["properties"]["continuidad"] = "Interrumpida (Conexión < 20m)"
            feature_actual["properties"]["caminable_potencial"] = "Sí"
        else:
            feature_actual["properties"]["continuidad"] = "Aislada (Sin conexiones cercanas)"
            feature_actual["properties"]["caminable_potencial"] = "No"
            
        elementos_validos.append(feature_actual)
    
    zonas_geojson["features"] = elementos_validos
    return zonas_geojson

In [7]:
# Build query
# City's bounding box (extreme points defining a circumscribed rectangle): 
# south, west, north, east
#CITY_BBOX = (4.46, -74.22, 4.84, -73.99) # Bogotá's bounding box
CITY_BBOX = (4.64, -74.09, 4.69, -74.06) # Localidad de  Barrios Unidos
#CITY_BBOX = (4.63, -74.19, 4.64, -74.18) # Parque el porvenir
CITY_NAME = "Bogotá"
CITY_COUNTRY = "Colombia"

In [8]:
query=build_query(CITY_NAME, CITY_COUNTRY, CITY_BBOX, QUERY_TIME_OUT)
print(query)


    [out:json][timeout:600];
    (
        way["highway"="pedestrian"]["bridge"!~"."](4.64,-74.09,4.69,-74.06);
        way["highway"="footway"]["footway"!~"^(sidewalk|crossing|traffic_island|link)$"]["bridge"!~"."](4.64,-74.09,4.69,-74.06);
        way["highway"="path"]["footway"!~"^(sidewalk|crossing)$"]["bridge"!~"."](4.64,-74.09,4.69,-74.06);
        way["leisure"="park"](4.64,-74.09,4.69,-74.06);
        relation["leisure"="park"](4.64,-74.09,4.69,-74.06);
    );
    out body geom;
    


In [9]:
fetched_data = run_query(query, QUERY_TIME_OUT, HEADERS)

[query] Sending Overpass query (attempt 1/3)...


In [10]:
total = len([e for e in fetched_data["elements"] if e["type"] in ["way", "relation"]])
print(f"[query] Retrieved {total} pedestrian zones and parks")
zones_geojson = overpass_to_geojson(fetched_data)
zones_geojson = identificar_zonas_interrumpidas(zones_geojson, radio_metros=25)
with_zones = sum(1 for f in zones_geojson["features"] if f["properties"]["superficie"])
print(f"[query] {with_zones}/{total} zonas que tienen 'superficie' tag "
        f"({100 * with_zones / total:.1f}%)")

[query] Retrieved 1484 pedestrian zones and parks
[Análisis] Procesando 1023 elementos peatonales
[query] 184/1484 zonas que tienen 'superficie' tag (12.4%)


In [11]:
with open("zonas_peatonales_analizadas.geojson", "w", encoding="utf-8") as f:
        json.dump(zones_geojson, f, ensure_ascii=False, indent=2)

In [17]:
def exportar_a_shapefile(geojson_data):
    nombre_carpeta = "zonas_caminables_shp"
    nombre_salida = "zonas_caminables_bogota"
   
    """
    Convierte el diccionario GeoJSON analizado en un Shapefile
    y crea una carpeta para guardar los archivos.
    """
    try:
        #1. Crea la carpeta
        if not os.path.exists(nombre_carpeta):
            os.makedirs(nombre_carpeta)
            print(f"[Carpeta] Carpeta creada con éxito'{nombre_carpeta}'")
        else: 
            print(f"[Carpeta] la carpeta '{nombre_carpeta}' ya existe")
            # Limpia el contenido de la carpeta
            for archivo in os.listdir(nombre_carpeta):
                ruta_archivo = os.path.join(nombre_carpeta, archivo)
                if os.path.isfile(ruta_archivo):
                    os.unlink(ruta_archivo)
        
        print("[Shapefile] Convirtiendo datos a DataFrame espacial")
        # 2. Carga el GeoJSON directamente en un GeoDataFrame de GeoPandas
        gdf = gpd.GeoDataFrame.from_features(geojson_data["features"])
        
        # 2. Le asigna el sistema de coordenadas geográficas estándar (WGS84)
        gdf.set_crs(epsg=4326, inplace=True)
        
        # 3. Guarda el paquete de archivos Shapefile
        archivo_shp = os.path.join(nombre_carpeta, f"{nombre_salida}.shp")
        gdf.to_file(archivo_shp, driver="ESRI Shapefile", encoding="utf-8")
        print(f"[Shapefile] ¡Éxito! Archivo guardado como '{archivo_shp}'")
        
    except Exception as e:
        print(f"[Error Shapefile] No se pudo exportar: {e}")

# --- EJECUCIÓN ---
exportar_a_shapefile(zones_geojson)

[Carpeta] la carpeta 'zonas_caminables_shp' ya existe
[Shapefile] Convirtiendo datos a DataFrame espacial


C:\Users\jessi\AppData\Local\Temp\ipykernel_15588\3995214948.py:31: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(archivo_shp, driver="ESRI Shapefile", encoding="utf-8")


[Shapefile] ¡Éxito! Archivo guardado como 'zonas_caminables_shp\zonas_caminables_bogota.shp'


C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'iluminacion' to 'iluminacio'
  ogr_write(
C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'sillasruedas' to 'sillasrued'
  ogr_write(
C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'tiposendero' to 'tiposender'
  ogr_write(
C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'continuidad' to 'continuida'
  ogr_write(
C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'caminable_potencial' to 'caminable_'
  ogr_write(
C:\Users\jessi\Downloads\_Prueba_Zonas_Peatonales\.venv\lib\s